#### Instead of standard absa and making the model extract the aspects and give a sentiment here the aspects are given but the model needs to give a sentiment conditioned Aspect Based Sentiment Analysis

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import torch
print("CUDA available:", torch.cuda.is_available())

CUDA available: True


In [3]:

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_path = "drive/MyDrive/FYP/ALLAM-7B"

tokenizer = AutoTokenizer.from_pretrained(model_path)

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.float16,
    device_map="auto"
)
model.eval()



`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:08<?, ?it/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(64000, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (v_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=4096, out_features=11008, bias=False)
          (up_proj): Linear(in_features=4096, out_features=11008, bias=False)
          (down_proj): Linear(in_features=11008, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((4096,), eps=1e-05)
   

In [18]:
def generate_chat(model, tokenizer, user_content: str):
    messages = [{"role": "user", "content": user_content}]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=40,
            do_sample=False,      # deterministic
            temperature=None,     # ignored when do_sample=False
            top_p=None,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id
        )
    gen_ids = out[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

In [5]:
import json
def extract_first_json(text):
    if not text:
        return None

    start = text.find("{")
    if start == -1:
        return None

    depth = 0
    for i in range(start, len(text)):
        ch = text[i]
        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                block = text[start:i+1]
                try:
                    return json.loads(block)
                except:
                    return None
    return None

##### Changed the Extract First Json because it couldnt track the {...} block because it was nested in here

In [6]:
def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows

In [7]:
absa_train_path = "/content/drive/MyDrive/FYP/absa/absa_train_big.jsonl"
absa_dev_path = "/content/drive/MyDrive/FYP/absa/absa_dev.jsonl"
absa_test_path = "/content/drive/MyDrive/FYP/absa/absa_test.jsonl"

absa_train = load_jsonl(absa_train_path)
absa_dev = load_jsonl(absa_dev_path)
absa_test = load_jsonl(absa_test_path)

In [19]:
ABSA_POLARITIES = {"positive", "negative", "neutral", "conflict"}
ABSA_POLARITY_SET = set(ABSA_POLARITIES)

In [25]:
def build_conditioned_absa_prompt(text, category):
    return (
        "أنت نظام تصنيف فقط، وليس مساعداً.\n"
        "مهمتك الوحيدة هي اختيار قيمة polarity صحيحة.\n\n"

        "سيتم إعطاؤك:\n"
        "- نص مراجعة فندق واحد\n"
        "- جانب واحد فقط (category)\n\n"

        "المطلوب:\n"
        "اختيار polarity هذا الجانب فقط.\n\n"

        "القيم المسموحة حصراً (اكتبها كما هي بالإنجليزية):\n"
        "positive | negative | neutral | conflict\n\n"

        "قواعد إلزامية صارمة:\n"
        "1) لا تكتب أي شرح أو تفسير أو تعليق.\n"
        "2) لا تذكر النص مرة أخرى.\n"
        "3) لا تذكر اسم الـ category.\n"
        "4) لا تستخدم كلمات عربية للمشاعر.\n"
        "5) لا تضف أي نص خارج JSON.\n"
        "6) الإخراج يجب أن يكون JSON فقط وعلى سطر واحد.\n"
        "7) يجب أن يبدأ الإخراج بـ { وينتهي بـ }.\n"
        "8) أي مخالفة لهذه القواعد تعتبر خطأ.\n\n"

        "صيغة الإخراج المطلوبة حرفياً:\n"
        "{\"labels\":[{\"polarity\":\"positive\"}]}\n\n"

        f"category: {category}\n"
        f"text: {text}\n\n"
        "output:\n"
    )

In [21]:
def parse_polarity_from_obj(obj):
    try:
        pol = obj["labels"][0]["polarity"]
        if isinstance(pol, str):
            pol = pol.strip().lower().replace(" ", "").replace("▁", "")
        return pol if pol in ABSA_POLARITY_SET else None
    except Exception:
        return None

def parse_polarity_from_text(raw):
    if not raw:
        return None
    s = raw.lower().replace(" ", "").replace("▁", "")
    for pol in ABSA_POLARITIES:
        if pol in s:
            return pol
    return None

def predict_conditioned_polarity(raw, category):
    obj = extract_first_json(raw)
    pol = parse_polarity_from_obj(obj) if obj else None
    if pol is None:
        pol = parse_polarity_from_text(raw)

    if pol is None:
        return None
    return (category, pol)

In [12]:
from collections import defaultdict

def macro_f1_polarity(y_true, y_pred, labels=ABSA_POLARITIES):
    tp = defaultdict(int)
    fp = defaultdict(int)
    fn = defaultdict(int)

    for tset, pset in zip(y_true, y_pred):
        true_map = {c:p for c,p in tset}
        pred_map = {c:p for c,p in pset}

        for c, true_pol in true_map.items():
            pred_pol = pred_map.get(c)
            if pred_pol == true_pol:
                tp[true_pol] += 1
            else:
                fn[true_pol] += 1
                if pred_pol is not None:
                    fp[pred_pol] += 1

    f1s = []
    for pol in labels:
        p = tp[pol] / (tp[pol] + fp[pol]) if (tp[pol] + fp[pol]) else 0.0
        r = tp[pol] / (tp[pol] + fn[pol]) if (tp[pol] + fn[pol]) else 0.0
        f1 = (2*p*r)/(p+r) if (p+r) else 0.0
        f1s.append(f1)

    return sum(f1s) / len(f1s)


In [13]:
def polarity_accuracy(y_true, y_pred):
    correct = total = 0
    for tset, pset in zip(y_true, y_pred):
        true_map = {c:p for c,p in tset}
        pred_map = {c:p for c,p in pset}
        for c, tp in true_map.items():
            total += 1
            if pred_map.get(c) == tp:
                correct += 1
    return correct / total if total else 0.0

In [14]:
from tqdm import tqdm

In [22]:
def eval_absa_conditioned(rows):
    bad = 0
    y_pred, y_true = [], []

    for r in tqdm(rows):
        gold_pairs = [(x["category"], x["polarity"]) for x in r["labels"]]
        gold_cats = sorted({cat for cat, _ in gold_pairs})

        pred_set = set()

        for cat in gold_cats:
            prompt = build_conditioned_absa_prompt(r["text"], cat)

            raw = generate_chat(model, tokenizer, prompt)

            print("\nRAW:", raw)

            pred_item = predict_conditioned_polarity(raw, cat)
            print(pred_item)
            if pred_item is None:
                bad += 1
                continue

            pred_set.add(pred_item)

        y_true.append(set(gold_pairs))
        y_pred.append(pred_set)

    return {
        "macro-f1": macro_f1_polarity(y_true, y_pred),
        "accuracy": polarity_accuracy(y_true, y_pred),
        "invalid": bad,
        "n": len(rows),
    }

In [26]:
zeroS = eval_absa_conditioned(absa_test)

  0%|          | 0/452 [00:00<?, ?it/s]


RAW: ▁{" lab els ": [ {" pol ar ity ":" pos it ive "} ] ," ca te go ry ":" F AC IL IT IES "} ▁
('FACILITIES#PRICES', 'positive')

RAW: ▁{" lab els ": [ {" pol ar ity ":" pos it ive "} ] ," ca te go ry ":" H OT EL # GEN ER AL "} ▁
('HOTEL#GENERAL', 'positive')


  0%|          | 1/452 [00:01<11:23,  1.52s/it]


RAW: ▁{" lab els ": [ {" pol ar ity ":" pos it ive "} ] ," ca te go ry ": ▁" L OC AT ION "} ▁
('LOCATION#GENERAL', 'positive')

RAW: ▁{" lab els ": [ {" pol ar ity ":" pos it ive "} ] ," ca te go ry ":" H OT EL # GEN ER AL "} ▁
('HOTEL#GENERAL', 'positive')


  0%|          | 2/452 [00:02<09:08,  1.22s/it]

...


RAW: ▁{" lab els ": [ {" pol ar ity ":" neg at ive "} ] ," ca ▁te go ▁r y ": ▁" S ER VI CE # GEN ER AL "} ▁
('SERVICE#GENERAL', 'negative')

RAW: ▁{" lab els ": [ {" pol ar ity ":" neg at ive "} ] ," ca te go ry ":" F AC IL IT IES # GEN ER AL "} ▁
('FACILITIES#GENERAL', 'negative')

RAW: ▁{" lab els ": [ {" pol ar ity ":" neg at ive "} ] ," ca te go ry ":" F AC IL IT IES "} ▁
('FACILITIES#MISCELLANEOUS', 'negative')

RAW: ▁{" lab els ": [ {" pol ar ity ":" neg at ive "} ] ," ca te go ry ":" H OT EL # GEN ER AL "} ▁
('HOTEL#GENERAL', 'negative')

RAW: ▁{" lab els ": [ {" pol ar ity ":" neg at ive "} ] ," ca te go ry ":" L OC AT ION # GEN ER AL "} ▁
('LOCATION#GENERAL', 'negative')

RAW: ▁{" lab els ": [ {" pol ar ity ":" neg at ive "} ] ," ca te go ry ":" R OO MS # GEN ER AL "} ▁
('ROOMS#GENERAL', 'negative')

RAW: ▁{" lab els ": [ {" pol ar ity ":" neg at ive "} ] ," ca te go ry ": ▁" R OO MS _ AM EN IT IES # DE SI GN _ FE AT UR ES "}
('ROOMS_AMENITIES#DESIGN_FEATURES', 'negative')

R

100%|█████████▉| 451/452 [19:31<00:03,  3.64s/it]


RAW: ▁{" lab els ": [ {" pol ar ity ":" neg at ive "} ] ," ca te go ry ":" S ER VI CE # GEN ER AL "} ▁
('SERVICE#GENERAL', 'negative')

RAW: ▁{" lab els ": [ {" pol ar ity ":" pos it ive "} ] ," ca ▁te go ▁r y ": ▁" H OT EL # CL EA ▁NL IN ESS "} ▁
('HOTEL#CLEANLINESS', 'positive')

RAW: ▁{" lab els ": [ {" pol ar ity ":" pos it ive "} ] ," ca ▁te go ▁r y ": ▁" H OT EL # GEN ER AL "} ▁
('HOTEL#GENERAL', 'positive')


100%|██████████| 452/452 [19:33<00:00,  2.60s/it]


RAW: ▁{" lab els ": [ {" pol ar ity ":" pos it ive "} ] ," ca te go ry ":" S ER VI CE # GEN ER AL "} ▁
('SERVICE#GENERAL', 'positive')


In [27]:
print(zeroS)

{'macro-f1': 0.3846656976744186, 'accuracy': 0.7215013901760889, 'invalid': 0, 'n': 452}


#Analysis
Although the model achieves relatively high accuracy (72.1%), the macro-averaged F1 score is considerably lower (0.38). This discrepancy indicates a strong class imbalance and a tendency to over-predict the dominant positive polarity, which is a known limitation of large language models in zero-shot sentiment classification. Performance on minority classes such as neutral and conflict remains challenging.

In [28]:
results = {
    "zeroS": zeroS,
}

In [29]:
results_path = "drive/MyDrive/FYP/absa/prompt_engineering_results/conditioned_absa_test_eval_results.json"
with open(results_path, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False)

We frame ABSA polarity prediction as a conditioned classification task. For each review and gold aspect category, the model is prompted to predict only the sentiment polarity. The prompt enforces a strict JSON-only output format to enable deterministic parsing. Any outputs that violate the format are counted as invalid predictions. Evaluation is performed using macro-averaged F1 and accuracy across polarity classes.